# ADR / ACCE / EPT - Data Products + Preview Sample

End-to-end pipeline that mirrors the `data-quality-app` repo pattern:

1. **Connect to Snowflake** via `snowflake.connector` with `externalbrowser` auth
   (or use `DATA_SOURCE=mock` for local dev).
2. **Build each data product** by loading tables and left-joining them on the
   estimate-item PK (single-table systems are used as-is). `PROJECT_KEY` is
   preserved but not used in the join.
3. **Coverage-sample** the resulting data product so an AI agent can preview
   it faithfully (top-N values per categorical column, quantiles per numeric
   column, min/max per datetime, nulls represented).
4. **Save** `<system>_data_product.parquet`, `<system>_preview.csv` and
   `<system>_summary.json` for each system.

### Requirements
```
pip install snowflake-connector-python[pandas] python-dotenv pandas numpy pyarrow
```

### `.env` (same keys as the repo)
```
DATA_SOURCE=snowflake          # or "mock"
SNOWFLAKE_ACCOUNT=...
SNOWFLAKE_USER=...
SNOWFLAKE_AUTHENTICATOR=externalbrowser
SNOWFLAKE_WAREHOUSE=...
SNOWFLAKE_DATABASE=...
SNOWFLAKE_SCHEMA=...
SNOWFLAKE_ROLE=                # optional
MAX_ROWS_PER_TABLE=100000
```


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

## Config

Edit `SYSTEMS` to match your real table names in Snowflake. The shape mirrors
`config/systems.py` in the `data-quality-app` repo: each system lists its
tables (with `primary` being the one whose PK drives the join), the join key,
and the join type. Single-table systems just declare the one table.

In [ ]:
# ----- Paths -----
OUTPUT_DIR = Path("data_products")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----- Runtime toggles -----
DATA_SOURCE  = os.getenv("DATA_SOURCE", "mock").lower()       # "mock" or "snowflake"
MAX_ROWS     = int(os.getenv("MAX_ROWS_PER_TABLE", "100000")) # applied per table
RANDOM_STATE = 42

# ----- Systems definition -----
# TODO: replace placeholder table names with the real ones in your Snowflake
# schema. Adjust join_key if your PKs use a different name.
SYSTEMS: dict = {
    "ADR": {
        "tables": {
            "primary": "ADR_ESTIMATEITEMRECORD",          # PK = ROW_ID
            "cost":    "ADR_FACT_ESTIMATECOSTRESULTS",
            "qty":     "ADR_FACT_ESTIMATEQTYRESULTS",
        },
        "join_key":  "ROW_ID",
        "join_type": "left",
    },
    "ACCE": {
        "tables": {
            "primary": "ACCE_ESTIMATEITEMRECORD",
            "cost":    "ACCE_ESTIMATECOSTRESULTS",
            "qty":     "ACCE_ESTIMATEQTYRESULTS",
        },
        "join_key":  "ROW_ID",
        "join_type": "left",
    },
    "EPT": {
        # single-table system - no join needed
        "tables": {"primary": "EPT_REFDATA"},
        "join_key":  None,
        "join_type": None,
    },
}

# ----- Sampling parameters -----
MAX_CARDINALITY   = 50
TOP_N             = 10
NUMERIC_QUANTILES = [0.0, 0.25, 0.5, 0.75, 1.0]
INCLUDE_NULLS     = True
MIN_SAMPLE_SIZE   = 50

## Snowflake connection

Same pattern as `src/snowflake_client.py` in the repo - reads env vars and
connects via `externalbrowser` (a browser window will open on first run for
SSO login).

In [ ]:
def get_snowflake_connection():
    """Open a Snowflake connection using env vars and externalbrowser auth."""
    import snowflake.connector

    return snowflake.connector.connect(
        account       = os.environ["SNOWFLAKE_ACCOUNT"],
        user          = os.environ["SNOWFLAKE_USER"],
        authenticator = os.getenv("SNOWFLAKE_AUTHENTICATOR", "externalbrowser"),
        warehouse     = os.environ["SNOWFLAKE_WAREHOUSE"],
        database      = os.environ["SNOWFLAKE_DATABASE"],
        schema        = os.environ["SNOWFLAKE_SCHEMA"],
        role          = os.getenv("SNOWFLAKE_ROLE") or None,
    )


def load_table(conn, table: str, max_rows: Optional[int] = None) -> pd.DataFrame:
    """SELECT * FROM <table> [LIMIT N] and return as a pandas DataFrame."""
    sql = f"SELECT * FROM {table}"
    if max_rows:
        sql += f" LIMIT {int(max_rows)}"
    cur = conn.cursor()
    try:
        cur.execute(sql)
        df = cur.fetch_pandas_all()
    finally:
        cur.close()
    return df

## Mock mode (`DATA_SOURCE=mock`)

Synthetic data for local dev - no Snowflake credits consumed. Mirrors the
shape of cost-estimate tables so the full pipeline can be exercised end-to-end.

In [ ]:
def mock_data_product(system: str, n_rows: int = 5_000, seed: int = 0) -> pd.DataFrame:
    """Generate a synthetic data product for demo/dev mode."""
    rng = np.random.default_rng(hash(system) % (2**32 - 1) + seed)
    n = n_rows
    df = pd.DataFrame({
        "ROW_ID":       [f"{system}-{i:06d}" for i in range(n)],
        "PROJECT_KEY":  rng.choice([f"PRJ-{i:03d}" for i in range(50)], size=n),
        "DISCIPLINE":   rng.choice(["Civil", "Mech", "Elec", "Instr", "Piping"], size=n),
        "COMMODITY":    rng.choice([f"CMD-{i:02d}" for i in range(120)], size=n),  # high-cardinality
        "STATUS":       rng.choice(["draft","approved","rejected","pending"],
                                   size=n, p=[.3,.4,.2,.1]),
        "CURRENCY":     rng.choice(["USD","BRL","EUR"], size=n, p=[.6,.3,.1]),
        "UNIT_COST":    rng.lognormal(8, 1, size=n),
        "QTY":          rng.lognormal(3, 1, size=n),
        "TOTAL_COST":   None,
        "YEAR":         rng.integers(2018, 2026, size=n),
        "CREATED_AT":   pd.to_datetime("2020-01-01")
                        + pd.to_timedelta(rng.integers(0, 2000, size=n), unit="D"),
    })
    df["TOTAL_COST"] = df["UNIT_COST"] * df["QTY"]
    # inject a few nulls so the sampler picks them up
    df.loc[rng.choice(df.index, size=max(1, n // 100), replace=False), "UNIT_COST"] = np.nan
    df.loc[rng.choice(df.index, size=max(1, n // 200), replace=False), "CURRENCY"]  = np.nan
    return df

## Data product builder

For multi-table systems (ADR / ACCE): load the primary table, then left-join
the others on the configured key. For single-table systems (EPT): load directly.
Mock mode bypasses this and returns a synthetic data product per system.

In [ ]:
def build_data_product(
    system: str,
    config: dict,
    *,
    conn=None,
    source: str = "snowflake",
    max_rows: int = 100_000,
) -> pd.DataFrame:
    """Return the joined data product for `system`."""
    if source == "mock":
        return mock_data_product(system, n_rows=max_rows if max_rows < 20_000 else 10_000)

    if conn is None:
        raise ValueError("Snowflake connection required when source='snowflake'")

    tables    = config["tables"]
    join_key  = config.get("join_key")
    join_type = config.get("join_type", "left")

    # single-table system
    if len(tables) == 1 or join_key is None:
        name = tables["primary"] if "primary" in tables else next(iter(tables.values()))
        print(f"  [{system}] loading {name} (single table)")
        return load_table(conn, name, max_rows)

    # multi-table: primary first, then join the others
    primary = tables["primary"]
    print(f"  [{system}] loading primary: {primary}")
    product = load_table(conn, primary, max_rows)

    for alias, name in tables.items():
        if alias == "primary":
            continue
        print(f"  [{system}] joining {name} on {join_key} ({join_type})")
        other = load_table(conn, name, max_rows)
        product = product.merge(
            other,
            on=join_key,
            how=join_type,
            suffixes=("", f"_{alias}"),
        )

    return product

## Coverage sampler + column summary

Reused from the previous notebook. The sampler guarantees the top-N values of
every low/mid-cardinality categorical column appear in the sample, covers
numeric quantile boundaries and datetime min/max, and preserves at least one
null per column that has nulls. High-cardinality columns are skipped for
targeted coverage but still appear via random padding.

In [ ]:
def stratified_preview_sample(
    df: pd.DataFrame,
    max_cardinality: int = 50,
    top_n: int = 10,
    numeric_quantiles: list | None = None,
    include_nulls: bool = True,
    min_size: int = 0,
    random_state: int = 42,
) -> tuple[pd.DataFrame, dict]:
    """Return (sample_df, report). See previous notebook for full docs."""
    if numeric_quantiles is None:
        numeric_quantiles = [0.0, 0.25, 0.5, 0.75, 1.0]
    rng = np.random.default_rng(random_state)

    cat_cols, num_cols, dt_cols, skipped_cat = [], [], [], []
    for col in df.columns:
        s = df[col]
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            num_cols.append(col)
        elif pd.api.types.is_datetime64_any_dtype(s):
            dt_cols.append(col)
        else:
            (cat_cols if s.nunique(dropna=True) <= max_cardinality else skipped_cat).append(col)

    targets: dict[tuple, set] = {}

    for col in cat_cols:
        for val in df[col].value_counts(dropna=True).head(top_n).index:
            rows = set(df.index[df[col] == val])
            if rows:
                targets[(col, f"val={val!r}")] = rows

    for col in num_cols:
        s = df[col].dropna()
        if s.empty:
            continue
        for q in numeric_quantiles:
            target = s.quantile(q)
            closest = (df[col] - target).abs().nsmallest(1).index
            targets[(col, f"q={q}")] = set(closest)

    for col in dt_cols:
        s = df[col].dropna()
        if s.empty:
            continue
        targets[(col, "min")] = {s.idxmin()}
        targets[(col, "max")] = {s.idxmax()}

    if include_nulls:
        for col in df.columns:
            null_rows = set(df.index[df[col].isna()])
            if null_rows:
                targets[(col, "null")] = null_rows

    row_to_targets: dict = {}
    for key, rows in targets.items():
        for r in rows:
            row_to_targets.setdefault(r, set()).add(key)

    selected: set = set()
    uncovered = set(targets.keys())
    while uncovered:
        best_row, best_gain = None, 0
        for row, covers in row_to_targets.items():
            if row in selected:
                continue
            gain = len(covers & uncovered)
            if gain > best_gain:
                best_gain, best_row = gain, row
        if best_row is None:
            break
        selected.add(best_row)
        uncovered -= row_to_targets[best_row]

    if len(selected) < min_size:
        remaining = df.index.difference(list(selected))
        n_extra = min(min_size - len(selected), len(remaining))
        if n_extra > 0:
            extra = rng.choice(remaining.to_numpy(), size=n_extra, replace=False)
            selected.update(extra.tolist())

    sample_df = df.loc[sorted(selected)].reset_index(drop=True)
    report = {
        "original_rows": int(len(df)),
        "sample_rows": int(len(sample_df)),
        "compression_ratio": round(len(sample_df) / len(df), 6) if len(df) else None,
        "categorical_covered": cat_cols,
        "categorical_skipped_high_cardinality": skipped_cat,
        "numeric_covered": num_cols,
        "datetime_covered": dt_cols,
        "total_targets": len(targets),
        "uncovered_targets": len(uncovered),
    }
    return sample_df, report


def column_summary(df: pd.DataFrame, top_n: int = 10) -> dict:
    """Per-column metadata: dtype, unique, nulls, value_counts / describe."""
    summary = {}
    for col in df.columns:
        s = df[col]
        info = {
            "dtype": str(s.dtype),
            "n_unique": int(s.nunique(dropna=True)),
            "n_null": int(s.isna().sum()),
        }
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            desc = s.describe()
            info["describe"] = {k: (None if pd.isna(v) else float(v)) for k, v in desc.items()}
        elif pd.api.types.is_datetime64_any_dtype(s):
            info["min"] = None if s.dropna().empty else str(s.min())
            info["max"] = None if s.dropna().empty else str(s.max())
        else:
            vc = s.value_counts(dropna=False).head(top_n)
            info["top_values"] = {str(k): int(v) for k, v in vc.items()}
        summary[col] = info
    return summary

## Run the pipeline

For each system: build data product → coverage-sample → persist.
Generates three artifacts per system in `data_products/`:

- `<system>_data_product.parquet` - full joined data product (can be large)
- `<system>_preview.csv` - coverage sample ready to feed to an AI agent
- `<system>_summary.json` - column-level value_counts / describe companion

In [ ]:
conn = None
if DATA_SOURCE == "snowflake":
    print("Opening Snowflake connection (externalbrowser SSO)...")
    conn = get_snowflake_connection()
    print(f"  connected. warehouse={os.getenv('SNOWFLAKE_WAREHOUSE')} "
          f"db={os.getenv('SNOWFLAKE_DATABASE')} schema={os.getenv('SNOWFLAKE_SCHEMA')}")

reports: dict = {}
try:
    for name, config in SYSTEMS.items():
        print(f"\n=== {name} ===")
        dp = build_data_product(
            name, config,
            conn=conn, source=DATA_SOURCE, max_rows=MAX_ROWS,
        )
        print(f"  data product shape: {dp.shape}")

        # persist full data product (parquet handles dtypes and is much
        # smaller than CSV for typical cost-estimate data)
        dp_path = OUTPUT_DIR / f"{name.lower()}_data_product.parquet"
        dp.to_parquet(dp_path, index=False)

        sample, report = stratified_preview_sample(
            dp,
            max_cardinality=MAX_CARDINALITY,
            top_n=TOP_N,
            numeric_quantiles=NUMERIC_QUANTILES,
            include_nulls=INCLUDE_NULLS,
            min_size=MIN_SAMPLE_SIZE,
            random_state=RANDOM_STATE,
        )
        summary = column_summary(dp, top_n=TOP_N)

        sample_path  = OUTPUT_DIR / f"{name.lower()}_preview.csv"
        summary_path = OUTPUT_DIR / f"{name.lower()}_summary.json"
        sample.to_csv(sample_path, index=False)
        with summary_path.open("w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2, ensure_ascii=False, default=str)

        reports[name] = {**report, "data_product_shape": list(dp.shape)}
        print(f"  sample:  {report['original_rows']:,} -> {report['sample_rows']:,} rows "
              f"(ratio {report['compression_ratio']})")
        print(f"  covered cat: {len(report['categorical_covered'])} | "
              f"skipped high-card: {len(report['categorical_skipped_high_cardinality'])} | "
              f"numeric: {len(report['numeric_covered'])} | "
              f"datetime: {len(report['datetime_covered'])}")
        print(f"  -> {dp_path}")
        print(f"  -> {sample_path}")
        print(f"  -> {summary_path}")
finally:
    if conn is not None:
        conn.close()
        print("\nSnowflake connection closed.")

print("\nDone.")

## Sanity check

For the chosen system: confirm every top-N value of each covered categorical
column survived into the sample.

In [ ]:
SYSTEM_TO_CHECK = "ADR"   # change to "ACCE" or "EPT"

dp_full   = pd.read_parquet(OUTPUT_DIR / f"{SYSTEM_TO_CHECK.lower()}_data_product.parquet")
dp_sample = pd.read_csv(OUTPUT_DIR / f"{SYSTEM_TO_CHECK.lower()}_preview.csv", low_memory=False)

rows = []
for col in reports[SYSTEM_TO_CHECK]["categorical_covered"]:
    top_vals  = set(dp_full[col].value_counts(dropna=True).head(TOP_N).index.astype(str))
    in_sample = set(dp_sample[col].dropna().astype(str).unique())
    rows.append({
        "column": col,
        "top_n_expected": len(top_vals),
        "top_n_present": len(top_vals & in_sample),
        "missing": sorted(top_vals - in_sample)[:5],
    })

pd.DataFrame(rows).sort_values("top_n_present")